# Train a Point Cloud Classification Model using Point Transformer V3

[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/geoai/blob/main/docs/examples/train_point_cloud_ptv3.ipynb)

This notebook demonstrates how to train [Point Transformer V3](https://arxiv.org/abs/2312.10035) (PTv3) for semantic segmentation of aerial LiDAR point clouds using the [DALES dataset](https://udayton.edu/engineering/research/centers/vision_lab/research/was_702702702702702702702702702702/dales.php). A pre-trained checkpoint is available on [Hugging Face](https://huggingface.co/jayakumarpujar/Ptv3) for direct use or fine-tuning on custom aerial LiDAR data.

## Install packages

PTv3 requires a CUDA GPU and the [Pointcept](https://github.com/Pointcept/Pointcept) library with compiled CUDA extensions.

In [ ]:
# %pip install numpy "laspy[lazrs]>=2.5.0" tqdm

In [ ]:
import os, subprocess

# Clone geoai repo so scripts/ is available (Colab / fresh environment)
if not os.path.exists("geoai"):
    subprocess.run(
        ["git", "clone", "https://github.com/opengeos/geoai.git"], check=True
    )
os.chdir("geoai")

In [ ]:
import os
import subprocess
import sys

if not os.path.exists("Pointcept"):
    subprocess.run(
        ["git", "clone", "https://github.com/Pointcept/Pointcept.git"], check=True
    )
    subprocess.run(
        [sys.executable, "setup.py", "install"],
        cwd="Pointcept/libs/pointops",
        check=True,
    )

pointcept_root = os.path.abspath("Pointcept")
if pointcept_root not in sys.path:
    sys.path.insert(0, pointcept_root)

## Check GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu} ({mem:.1f} GB)")
    print(f"CUDA: {torch.version.cuda}")
else:
    print("No GPU detected. PTv3 requires CUDA.")

## Download and preprocess DALES

The [DALES dataset](https://udayton.edu/engineering/research/centers/vision_lab/research/was_702702702702702702702702702702/dales.php) contains 40 aerial LiDAR tiles (~500 m x 500 m each) with 9 semantic classes:

| ID | Class | ID | Class |
|---|---|---|---|
| 0 | Unknown (ignored) | 5 | Power lines |
| 1 | Ground | 6 | Fences |
| 2 | Vegetation | 7 | Poles |
| 3 | Cars | 8 | Buildings |
| 4 | Trucks | | |

Download and extract the dataset so that you have `dales_las/train/` (29 tiles) and `dales_las/test/` (11 tiles).

In [ ]:
## Train PTv3 model

The training script supports single-GPU and multi-GPU (DDP) training with cross-entropy + Lovász softmax loss, OneCycleLR scheduling, and automatic checkpoint management.

Adjust the flags based on your GPU:
- **V100 (32 GB)**: `--no_amp --max_points 8000`
- **A100 (80 GB)**: `--high_memory --flash_attn`
- **RTX 3090/4090 (24 GB)**: defaults work

## Train PTv3 model

The training script supports single-GPU and multi-GPU (DDP) training with cross-entropy + Lovasz softmax loss, OneCycleLR scheduling, and automatic checkpoint management.

Adjust the flags based on your GPU:
- **V100 (32 GB)**: `--no_amp --max_points 8000`
- **A100 (80 GB)**: `--high_memory --flash_attn`
- **RTX 3090/4090 (24 GB)**: defaults work

In [ ]:
!python scripts/train_ptv3_dales.py \
    --data_root data/dales_ptv3 \
    --save_dir checkpoints_ptv3 \
    --epochs 100 \
    --lr 0.0005 \
    --batch_size 1 --accum_steps 16 --num_workers 2 \
    --max_grad_norm 1.0 \
    --no_amp

For multi-GPU training:

```bash
torchrun --nproc_per_node=4 scripts/train_ptv3_dales.py \
    --data_root data/dales_ptv3 --epochs 100 --batch_size 2 --accum_steps 4
```

## Evaluate the model

In [ ]:
!python scripts/train_ptv3_dales.py \
    --data_root data/dales_ptv3 \
    --save_dir checkpoints_ptv3 \
    --eval_only \
    --checkpoint checkpoints_ptv3/ptv3_dales_best.pth \
    --no_amp \
    --batch_size 1 --num_workers 2

## Visualize training curves

In [ ]:
import json

import matplotlib.pyplot as plt

with open("checkpoints_ptv3/training_log.json") as f:
    log = json.load(f)

epochs = [e["epoch"] for e in log]
train_loss = [e["loss"] for e in log]
train_miou = [e["miou"] for e in log]
val_miou = [e["val_miou"] for e in log]

best = max(log, key=lambda e: e["val_miou"])
print(f"Best epoch: {best['epoch']}")
print(f"  Val mIoU:     {best['val_miou']:.4f}")
print(f"  Val accuracy: {best['val_acc']:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs, train_loss)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Training Loss")
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, train_miou, label="Train")
ax2.plot(epochs, val_miou, label="Val")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("mIoU")
ax2.set_title("Mean IoU")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Analyse confusion matrix

In [ ]:
import numpy as np

CLASSES = [
    "unknown",
    "ground",
    "vegetation",
    "cars",
    "trucks",
    "powerlines",
    "fences",
    "poles",
    "buildings",
]

for split in ("val", "test"):
    cm_path = f"checkpoints_ptv3/cm_{split}.npy"
    try:
        cm = np.load(cm_path)
    except FileNotFoundError:
        print(f"{split}: confusion matrix not found")
        continue

    print(f"\n{'=' * 60}")
    print(f"  {split.upper()} SET ({cm.sum():,.0f} points)")
    print(f"{'=' * 60}")
    print(f"{'Class':>12}  {'IoU':>7}  {'Prec':>7}  {'Recall':>7}  {'Support':>12}")
    print("-" * 55)

    ious = []
    for i in range(cm.shape[0]):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        support = int(cm[i, :].sum())
        print(f"{CLASSES[i]:>12}  {iou:7.4f}  {prec:7.4f}  {rec:7.4f}  {support:12,}")
        ious.append(iou)

    oa = np.trace(cm) / cm.sum()
    miou_8 = np.mean(ious[1:])
    print("-" * 55)
    print(f"  Overall Accuracy:  {oa:.4f}")
    print(f"  mIoU (8 classes):  {miou_8:.4f}")

## Use pre-trained checkpoint

A pre-trained PTv3 checkpoint trained on DALES is available on [Hugging Face](https://huggingface.co/jayakumarpujar/Ptv3) for direct inference or fine-tuning on custom aerial LiDAR data.

In [ ]:
# %pip install huggingface_hub

In [ ]:
from huggingface_hub import hf_hub_download

ckpt_path = hf_hub_download(
    repo_id="jayakumarpujar/Ptv3",
    filename="base_model_ptv3_dales_.pth",
)

In [ ]:
# Evaluate the pre-trained model on your data
!python scripts/train_ptv3_dales.py \
    --data_root data/dales_ptv3 \
    --eval_only \
    --checkpoint $ckpt_path \
    --no_amp

In [ ]:
# Fine-tune the pre-trained model on custom data (auto-downloads from HuggingFace)
!python scripts/train_ptv3_dales.py \
    --data_root /path/to/your/data \
    --hf_pretrained \
    --epochs 50 --lr 0.0001 \
    --no_amp

## References

- Wu, X. et al. "Point Transformer V3: Simpler, Faster, Stronger." CVPR 2024.
- Varney, N. et al. "DALES: A Large-scale Aerial LiDAR Data Set for Semantic Segmentation." CVPRW 2020.
- [Pointcept](https://github.com/Pointcept/Pointcept) -- PTv3 reference implementation.